<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day22_practice2_%EB%8F%84%EA%B5%AC%EC%99%80_%EB%A3%A8%ED%94%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 도구 3개 + Agent 루프
# 도구 3개 구현: 규정 검색 / 신고 조회 / 상태 조회
# Agent 루프 완성: 결정 → 실행 → 재질의 → 답(안전장치 포함)

In [4]:
import os, requests, json, re
import torch
import torch.nn.functional as F

def _get_api_key():
  try:
    from google.colab import userdata
    return userdata.get("VISION_API_KEY")
  except Exception:
    return os.environ.get("VISION_API_KEY", "") # Colab이 아니거나 Secret이 없을 때 환경변수 딕셔너리에서 꺼냄

API_KEY = _get_api_key()
# API 호출 - openAI 호환 api 형식으로 코드 작성
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai" # Gemini OpenAI 호환
MODEL = "gemini-3.5-flash-lite"

In [5]:
# 셀 1. 신고DB
REPORTS_DB = [
    {"id": 12, "hazard_type": "포트홀", "severity": 4, "location": "강남대로 사거리", "status": "처리중"},
    {"id": 15, "hazard_type": "포트홀", "severity": 4, "location": "서초대로 12",     "status": "접수"},
    {"id": 18, "hazard_type": "포트홀", "severity": 3, "location": "충무로 30",     "status": "완료"},
    {"id": 21, "hazard_type": "균열",   "severity": 2, "location": "을지로 골목",   "status": "접수"},
    {"id": 25, "hazard_type": "맨홀",   "severity": 3, "location": "안국역 3번출구", "status": "처리중"},
]
KNOWLEDGE = [
    "심각도 4 이상의 포트홀은 접수 후 24시간 이내에 긴급 보수팀을 배정한다.",
    "심각도 3 이하의 도로 균열은 주간 정기 점검 일정에 포함하여 7일 이내 처리한다.",
    "맨홀 뚜껑 이상 신고는 안전 펜스를 우선 설치하고 상수도사업소에 이관한다.",
    "처리 완료된 신고는 사진과 함께 신고자에게 앱 알림으로 통보한다.",
]

In [6]:
# 셀 2. 도구 3개 (파이썬 함수)
def search_regulation(query: str) -> str: # 도구1 : 규정 검색
  sims = F.cosine_similarity(_embed(query, _vocab).unsqueeze(0). _doc_vecs)
  return KNOWLEDGE[int(sims.argmax())]

def query_reports(hazard_type=None, min_severity=None) -> dict: # 도구2 : 신고 조회
  rows = [r for r in REPORTS_DB
            if (hazard_type is None or r["hazard_type"] == hazard_type)
            and (min_severity is None or r["severity"] >= min_severity)]
  return {"count": len(rows), "items": [f"#{r['id']} {r['location']} (심각도{r['severity']}, {r['status']})" for r in rows]}

def get_report_status(report_id: int) -> str: # 도구3 : 상태 조회
  for r in REPORTS_DB:
    if r["id"] == int(report_id):
      return f"#{r['id']} {r['hazard_type']} ({r['location']}) - 현재 '{r['status']}'"
  return f"신고번호 {report_id}를 찾을 수 없습니다"